# 04 — Chốt và xử lý 202 ảnh synth bằng API

Ngày tạo: 08/10/2026. Theo yêu cầu mới, API làm reviewer cuối và pipeline áp dụng quyết định, không để cả 202 case ở pending. Phạm vi: 200 ảnh batch trước + 2 deferred, không đụng quyết định 20 ảnh pilot.

API `gpt-6.1-sol` reasoning medium nhận 4 ảnh: card, crop native, zoom cạnh và cutout. Quyết định giữ có giải trình hoặc tái ghép; nếu ownership/geometry vẫn mơ hồ thì tái ghép **thật** toàn bộ instance trong ảnh từ tài nguyên có sẵn. Bbox/area/segmentation RLE của ảnh mới lấy từ owner-mask được lưu, không đoán tọa độ sửa trên JPEG cũ.

Raw giữ nguyên. Output là COCO native cho đúng 202 ảnh, provenance/owner masks, task JSON theo mapping 5 task đã khóa và CSV review được cập nhật trong phạm vi 202 sau nghiệm thu. Release ảnh thật/holdout không thay đổi. Các bbox được giữ là quyết định review của AI, chưa có original owner-mask GT; bbox ảnh tái ghép có mask mới xác định.

[Cấu hình](../../configs/data/annotation_resolution_202_v1.json). [OpenAI Docs](https://developers.openai.com/api/docs/models/gpt-6.1-sol) / [Batch API](https://developers.openai.com/api/docs/guides/batch).

Các cờ mặc định tắt để Run All không tạo chi phí hoặc ghi quyết định: `RR_SUBMIT`, `RR_REFRESH`, `RR_APPLY`. Bật qua `AUTOCHECKOUT_RESOLUTION_SUBMIT=1`, `AUTOCHECKOUT_RESOLUTION_REFRESH=1`, `AUTOCHECKOUT_RESOLUTION_APPLY=1`. Cache/status bền vững; mỗi run có timestamp/config/log riêng. API có cửa sổ 24h; chạy lại từ receipts không tạo thêm batch.


In [ ]:
import os, sys, json, uuid, platform, subprocess, time, hashlib, csv
from pathlib import Path
from datetime import datetime
from collections import Counter
from PIL import Image

RR_REPO = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdp/main.py").exists()
)
if str(RR_REPO) not in sys.path:
    sys.path.insert(0, str(RR_REPO))
from tools.ai_annotation_review import file_sha256, write_json, load_api_key, timestamp
from tools.ai_annotation_batch import (
    BatchClient,
    prepare_bundle,
    submit_bundle,
    poll_bundle,
    parse_outputs,
)
from tools.annotation_resolution import (
    PROMPT,
    PROMPT_VERSION,
    build_adjudication_request,
    parse_adjudication,
    final_action,
)

RR_SUBMIT = os.getenv("AUTOCHECKOUT_RESOLUTION_SUBMIT", "0") == "1"
RR_REFRESH = os.getenv("AUTOCHECKOUT_RESOLUTION_REFRESH", "0") == "1"
RR_APPLY = os.getenv("AUTOCHECKOUT_RESOLUTION_APPLY", "0") == "1"
RR_WAIT_SECONDS = int(os.getenv("AUTOCHECKOUT_RESOLUTION_WAIT_SECONDS", "0"))
RR_RAW = Path(
    os.getenv("AUTOCHECKOUT_DATA_ROOT", str(RR_REPO / "data/archive"))
).resolve()
RR_REVIEW_ROOT = RR_REPO / "data/processed/synthetic_annotation_review_v1"
RR_OUT = RR_REVIEW_ROOT / "ai/gpt-6.1-sol_resolution_202_v1"
RR_OUT.mkdir(parents=True, exist_ok=True)
RR_DATASET = RR_REPO / "data/processed/rpc_synth_resolved_202_v1"
RR_SETTINGS = json.loads(
    (RR_REPO / "configs/data/annotation_resolution_202_v1.json").read_text()
)
assert (
    RR_SETTINGS["target_cases"] == 202
    and RR_SETTINGS["model"] == "gpt-6.1-sol"
    and RR_SETTINGS["prompt_version"] == PROMPT_VERSION
)
RR_ARTIFACT = (
    RR_REPO / "runs/data_preprocessing/annotation_resolution" / str(uuid.uuid4())
)
RR_ARTIFACT.mkdir(parents=True, exist_ok=True)
run = {
    "run_id": RR_ARTIFACT.name,
    "purpose": "Automated adjudication and actual resolution of 202 synthetic photos",
    "started_at": timestamp(),
    "finished_at": None,
    "timezone": "Asia/Bangkok",
    "python": sys.version.split()[0],
    "host": platform.node(),
    "cuda_executed": False,
    "settings": RR_SETTINGS,
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip(),
    "git_dirty": bool(
        subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
    ),
    "code_sha256": file_sha256(RR_REPO / "tools/annotation_resolution.py"),
    "status": "PREPARING",
}
RR_VISUAL_SUMMARY = json.loads((RR_REVIEW_ROOT / "review_summary.json").read_text())
RR_VISUAL = {
    (r["source_task"], r["annotation_id"]): r
    for r in json.loads((RR_REPO / RR_VISUAL_SUMMARY["visual_manifest"]).read_text())
}
RR_QUEUE = {
    (r["source_task"], r["annotation_id"]): r
    for r in json.loads((RR_REVIEW_ROOT / "review_queue.json").read_text())
}
RR_OLD_BATCH = json.loads(
    (RR_REVIEW_ROOT / "ai/gpt-6.1-sol_batch_200_v1/accepted_results.json").read_text()
)
RR_KEYS = sorted(
    {tuple(r["case_key"]) for r in RR_OLD_BATCH} | {(4, 57855), (4, 61176)}
)
assert len(RR_KEYS) == 202
RR_SOURCE_HASHES = {
    str(t): file_sha256(RR_RAW / f"synth/group/task{t}/annotations.json")
    for t in range(1, 5)
}
assert RR_SOURCE_HASHES == RR_VISUAL_SUMMARY["source_sha256"]
RR_ASSET_HASHES = {}
RR_CASES = []
RR_INPUT_IMAGES = RR_OUT / "input_images"
RR_INPUT_IMAGES.mkdir(exist_ok=True)
RR_MANUAL_BEFORE = file_sha256(RR_REVIEW_ROOT / "review_decisions.csv")
for key in RR_KEYS:
    q = RR_QUEUE[key]
    case = {**RR_VISUAL[key], **{k: q[k] for k in ["bbox", "polygon_bbox"]}}
    stem = f"task{key[0]}_ann{key[1]}"
    x, y, w, h = case["bbox"]
    px, py, pw, ph = case["polygon_bbox"]
    gaps = [px - x, py - y, x + w - px - pw, y + h - py - ph]
    edge = ["left", "top", "right", "bottom"][max(range(4), key=lambda i: abs(gaps[i]))]
    native = RR_INPUT_IMAGES / (stem + "_native.png")
    zoom = RR_INPUT_IMAGES / (stem + "_edge.png")
    with Image.open(RR_RAW / case["image_file"]) as im:
        if not native.exists():
            im.convert("RGB").crop(case["crop_box"]).save(native)
        if edge == "left":
            roi = (x - 16, y - 16, px + 16, y + h + 16)
        elif edge == "top":
            roi = (x - 16, y - 16, x + w + 16, py + 16)
        elif edge == "right":
            roi = (px + pw - 16, y - 16, x + w + 16, y + h + 16)
        else:
            roi = (x - 16, py + ph - 16, x + w + 16, y + h + 16)
        roi = (
            max(0, int(roi[0])),
            max(0, int(roi[1])),
            min(im.width, int(roi[2])),
            min(im.height, int(roi[3])),
        )
        if not zoom.exists():
            band = im.convert("RGB").crop(roi)
            band.resize(
                (band.width * 3, band.height * 3), Image.Resampling.NEAREST
            ).save(zoom)
    for name in [case["image_file"], case["cutout"]]:
        RR_ASSET_HASHES[name] = file_sha256(RR_RAW / name)
    assert file_sha256(RR_REPO / case["preview"]) == case["preview_sha256"]
    case.update(
        native_crop=str(native.resolve()),
        edge_zoom=str(zoom.resolve()),
        largest_gap_edge=edge,
        image_sha256=RR_ASSET_HASHES[case["image_file"]],
        edge_roi=list(roi),
    )
    RR_CASES.append(case)
RR_BACKGROUNDS = [
    p
    for p in sorted((RR_RAW / "backgrounds").glob("*.jpg"))
    if "preview" not in p.name.lower()
]
assert len(RR_BACKGROUNDS) == 208
RR_BACKGROUND_HASHES = {
    str(p.relative_to(RR_RAW)): file_sha256(p) for p in RR_BACKGROUNDS
}
RR_TASK_CONFIG = (
    RR_REPO / "data/processed/rpc_100-4x25_seed0_native_v1/task_config.json"
)
RR_TASK_CONFIG_HASH = file_sha256(RR_TASK_CONFIG)
RR_REAL_LOCK = (
    RR_REPO / "data/processed/rpc_100-4x25_seed0_native_v1/release_manifest.json"
)
RR_REAL_LOCK_HASH = file_sha256(RR_REAL_LOCK)
RR_MANIFEST = prepare_bundle(
    RR_CASES,
    repo=RR_REPO,
    raw_root=RR_RAW,
    out_dir=RR_OUT,
    max_bytes=RR_SETTINGS["max_jsonl_bytes"],
    request_builder=build_adjudication_request,
    prompt_version=PROMPT_VERSION,
    reasoning_effort=RR_SETTINGS["reasoning_effort"],
)
run.update(
    status="INPUT_PASS", cases=202, manifest_fingerprint=RR_MANIFEST["fingerprint"]
)
write_json(RR_ARTIFACT / "run.json", run)
write_json(RR_ARTIFACT / "selection.json", RR_CASES)
write_json(
    RR_ARTIFACT / "source_asset_sha256.json",
    {"raw": RR_ASSET_HASHES, "backgrounds": RR_BACKGROUND_HASHES},
)
(RR_ARTIFACT / "prompt.txt").write_text(PROMPT)
print(
    json.dumps(
        {"run": run, "shards": RR_MANIFEST["shards"], "input_gate": "PASS"},
        ensure_ascii=False,
        indent=2,
    )
)

{
  "run": {
    "run_id": "f3292ec6-ce43-47e5-a07f-9198d60b1e43",
    "purpose": "Automated adjudication and actual resolution of 202 synthetic photos",
    "started_at": "2026-10-08T13:31:19+07:00",
    "finished_at": null,
    "timezone": "Asia/Bangkok",
    "python": "3.14.4",
    "host": "Laptop",
    "cuda_executed": false,
    "settings": {
      "model": "gpt-6.1-sol",
      "prompt_version": "rpc-annotation-final-resolution-v1",
      "reasoning_effort": "medium",
      "image_detail": "high",
      "images_per_case": 4,
      "max_output_tokens": 4000,
      "target_cases": 202,
      "pricing_mode": "batch",
      "endpoint": "/v1/responses",
      "completion_window": "24h",
      "max_jsonl_bytes": 190000000,
      "poll_interval_seconds": 30,
      "seed": 20261008,
      "minimum_visible_fraction": 0.7,
      "decisions": [
        "keep_with_justification",
        "regenerate"
      ],
      "uncertain_action": "regenerate",
      "apply_resolutions": true,
      "writ

## 1. API reviewer cuối — đúng 202 case

4-view input và reasoning medium giúp API nhìn được các fragment mà card cũ thu nhỏ. JSON chỉ cho phép keep hoặc regenerate; một quyết định keep nhưng vẫn đánh dấu geometry_uncertain sẽ được policy chuyển sang regeneration. Không lấy self-confidence làm chứng nhận GT và không yêu cầu model đoán mask/bbox thay thế.


In [2]:
RR_STATE_PATH=RR_OUT/'submission.json'
RR_STATE=json.loads(RR_STATE_PATH.read_text()) if RR_STATE_PATH.exists() else None
if RR_SUBMIT or RR_REFRESH:
    RR_CLIENT=BatchClient(load_api_key(RR_REPO/'.env'))
    if RR_SUBMIT: RR_STATE=submit_bundle(RR_MANIFEST,out_dir=RR_OUT,client=RR_CLIENT)
    if RR_STATE:
        deadline=time.monotonic()+max(0,RR_WAIT_SECONDS)
        while True:
            RR_STATE=poll_bundle(RR_MANIFEST,out_dir=RR_OUT,client=RR_CLIENT)
            print(timestamp(),[(r['batch_id'],r['batch']['status'],r['batch'].get('request_counts')) for r in RR_STATE['shards']])
            write_json(RR_ARTIFACT/'submission_snapshot.json',RR_STATE)
            if RR_STATE['terminal'] or time.monotonic()>=deadline:break
            time.sleep(min(RR_SETTINGS['poll_interval_seconds'],max(0,deadline-time.monotonic())))
else: print('API disabled; use saved submission and downloaded outputs only.')
run['batch_ids']=[r['batch_id'] for r in RR_STATE['shards']] if RR_STATE else []
run['status']='BATCH_TERMINAL' if RR_STATE and RR_STATE.get('terminal') else ('BATCH_PENDING' if RR_STATE else 'NOT_SUBMITTED')
write_json(RR_ARTIFACT/'run.json',run)


API disabled; use saved submission and downloaded outputs only.


In [3]:
RR_ACCEPTED=[];RR_REJECTED=[];RR_API_PASS=False
if RR_STATE and RR_STATE.get('terminal'):
    rows=[]
    for shard in RR_STATE['shards']:
        for kind in ['output','error']:
            if shard.get(kind+'_path'):
                p=Path(shard[kind+'_path']);assert file_sha256(p)==shard[kind+'_sha256']
                rows.extend(json.loads(line) for line in p.read_text().splitlines() if line.strip())
    RR_ACCEPTED,RR_REJECTED=parse_outputs(RR_MANIFEST,rows,batch_ids=run['batch_ids'],response_parser=parse_adjudication)
    RR_API_PASS=len(RR_ACCEPTED)==202 and not RR_REJECTED and len(rows)==202
    write_json(RR_OUT/'accepted_results.json',RR_ACCEPTED);write_json(RR_OUT/'rejected_results.json',RR_REJECTED)
    write_json(RR_ARTIFACT/'accepted_results.json',RR_ACCEPTED);write_json(RR_ARTIFACT/'rejected_results.json',RR_REJECTED)
    for receipt in RR_ACCEPTED+RR_REJECTED:write_json(RR_OUT/'cases'/(receipt['custom_id']+'.json'),receipt)
RR_COST=sum(r['estimated_cost_usd'] or 0 for r in RR_ACCEPTED+RR_REJECTED)
write_json(RR_ARTIFACT/'api_acceptance.json',{'passed':RR_API_PASS,'valid':len(RR_ACCEPTED),'rejected':len(RR_REJECTED),
 'model_decisions':dict(Counter(r['review']['decision'] for r in RR_ACCEPTED)),
 'operational_actions':dict(Counter(final_action(r['review']) for r in RR_ACCEPTED)),
 'estimated_batch_cost_usd':RR_COST,'api_batch_ids':run['batch_ids']})
run['status']='API_PASS_READY_TO_APPLY' if RR_API_PASS else ('API_FAIL' if RR_REJECTED else run['status'])
write_json(RR_ARTIFACT/'run.json',run)
print('API gate:',RR_API_PASS,'accepted:',len(RR_ACCEPTED),'rejected:',len(RR_REJECTED),'cost:',RR_COST)


API gate: True accepted: 202 rejected: 0 cost: 1.00714975


## 2. Áp dụng thật: giữ hoặc tái ghép toàn bộ ảnh

Chỉ chạy khi đủ 202 adjudications hợp lệ. Bản mới giữ nguyên multiset SKU/instance của mỗi ảnh nguồn. Ảnh giữ dùng raw và bbox nguồn; ảnh regenerate dùng 208 nền đã kiểm tra + đúng cutout của mọi instance, lưu recipe/seed/transform và owner.npy/npz. Minimum visible fraction 0,70; layout fallback dạng grid khi placement chưa giữ đủ instance. Không flip sản phẩm để tránh chữ/logo bị soi gương.

Output segmentation ảnh regenerate là COCO RLE không nén, bảo toàn cả fragment nhỏ. Bbox và area lấy từ chính owner map. COCO native chung dùng image root là repository root, labels RPC; task JSON dùng mapping 5 task seed0 đã khóa. Việc đưa augmentation vào experiment là bước sau, không thay real-only release/test lock.


In [4]:
RR_CLOSURE=None
if RR_APPLY:
    assert RR_API_PASS,'Cannot apply incomplete API decisions'
    import importlib
    import tools.annotation_resolution as resolution_module
    resolution_module=importlib.reload(resolution_module)
    RR_CLOSURE=resolution_module.apply_resolution_dataset(RR_CASES,RR_ACCEPTED,repo=RR_REPO,raw_root=RR_RAW,
        out_dir=RR_DATASET,backgrounds=RR_BACKGROUNDS,seed=RR_SETTINGS['seed'],
        min_visible=RR_SETTINGS['minimum_visible_fraction'],task_config_path=RR_TASK_CONFIG)
    write_json(RR_ARTIFACT/'closure.json',RR_CLOSURE)
else: print('Apply disabled; no new dataset or decisions written by this run.')


Resolved 20 / 202
Resolved 40 / 202
Resolved 60 / 202
Resolved 80 / 202
Resolved 100 / 202
Resolved 120 / 202
Resolved 140 / 202
Resolved 160 / 202
Resolved 180 / 202
Resolved 200 / 202


## 3. Nghiệm thu, commit quyết định phạm vi 202 và kết thúc

Nghiệm thu đủ 202 ảnh / giữ đủ mọi object và 200 SKU, geometry bounds/native paths, RLE → owner mask → bbox/area, seed/provenance và task label range. Dataset tạo đủ và pass mới ghi CSV review: 202 scope được chốt với reviewer OpenAI và artifact evidence; giữ nguyên 20 dòng pilot ngoài phạm vi. Raw và real holdout giữ checksum. Mask GT mới chỉ thuộc ảnh regenerate; ảnh keep có bằng chứng review API, không giả định đã khôi phục original mask.


In [5]:
if RR_CLOSURE:
    RR_VERIFICATION=resolution_module.verify_resolution_dataset(RR_CLOSURE,repo=RR_REPO,raw_root=RR_RAW,out_dir=RR_DATASET)
    write_json(RR_ARTIFACT/'acceptance.json',RR_VERIFICATION)
    for task,h in RR_SOURCE_HASHES.items():assert file_sha256(RR_RAW/f'synth/group/task{task}/annotations.json')==h
    for path,h in RR_ASSET_HASHES.items():assert file_sha256(RR_RAW/path)==h
    for path,h in RR_BACKGROUND_HASHES.items():assert file_sha256(RR_RAW/path)==h
    assert file_sha256(RR_TASK_CONFIG)==RR_TASK_CONFIG_HASH and file_sha256(RR_REAL_LOCK)==RR_REAL_LOCK_HASH
    (RR_ARTIFACT/'review_decisions_before.csv').write_bytes((RR_REVIEW_ROOT/'review_decisions.csv').read_bytes())
    resolution_module.update_review_decisions(RR_REVIEW_ROOT/'review_decisions.csv',RR_CLOSURE['resolutions'],
        reviewer='OpenAI/gpt-6.1-sol/'+PROMPT_VERSION)
    final_csv=list(csv.DictReader((RR_REVIEW_ROOT/'review_decisions.csv').open()))
    scope=set(RR_KEYS)
    assert all(r['decision']!='pending' and r['evidence'].strip() and r['reviewer'].strip()
               for r in final_csv if (int(r['source_task']),int(r['annotation_id'])) in scope)
    RR_VISUAL_SUMMARY.update(reviewed_cases_with_evidence=sum(r['decision']!='pending' for r in final_csv),
        repair_acceptance='PASS_FOR_RESOLVED_SCOPE_202',resolved_scope_cases=202,
        outside_scope_pending=sum(r['decision']=='pending' for r in final_csv),
        resolution_manifest=str((RR_DATASET/'resolution_manifest.json').relative_to(RR_REPO)),
        decisions_after_sha256=file_sha256(RR_REVIEW_ROOT/'review_decisions.csv'),source_modified=False)
    write_json(RR_REVIEW_ROOT/'review_summary.json',RR_VISUAL_SUMMARY)
    run['status']='RESOLVED_202_ACCEPTANCE_PASS'
else: print('Resolution acceptance pending until API complete and apply enabled.')
run['finished_at']=timestamp()
run['duration_seconds']=(datetime.fromisoformat(run['finished_at'])-datetime.fromisoformat(run['started_at'])).total_seconds()
RR_COMPACT_CLOSURE={k:RR_CLOSURE[k] for k in ['resolved_cases','kept_images','regenerated_images','object_count','sku_count','dataset','dataset_sha256']} if RR_CLOSURE else None
RR_SUMMARY={'run':run,'api_acceptance':RR_API_PASS,'closure':RR_COMPACT_CLOSURE,'estimated_batch_cost_usd':RR_COST,
 'source_unchanged':True,'real_release_unchanged':True,'original_owner_masks_recovered':False,
 'adjudication_origin':'OpenAI/gpt-6.1-sol','scope_cases':202,
 'dataset_acceptance':RR_VERIFICATION if RR_CLOSURE else None,
 'decision_csv_before_sha256':RR_MANUAL_BEFORE,'decision_csv_after_sha256':file_sha256(RR_REVIEW_ROOT/'review_decisions.csv')}
write_json(RR_ARTIFACT/'run.json',run);write_json(RR_ARTIFACT/'summary.json',RR_SUMMARY);write_json(RR_OUT/'latest_summary.json',RR_SUMMARY)
print(json.dumps(RR_SUMMARY,ensure_ascii=False,indent=2))


{
  "run": {
    "run_id": "f3292ec6-ce43-47e5-a07f-9198d60b1e43",
    "purpose": "Automated adjudication and actual resolution of 202 synthetic photos",
    "started_at": "2026-10-08T13:31:19+07:00",
    "finished_at": "2026-10-08T13:34:20+07:00",
    "timezone": "Asia/Bangkok",
    "python": "3.14.4",
    "host": "Laptop",
    "cuda_executed": false,
    "settings": {
      "model": "gpt-6.1-sol",
      "prompt_version": "rpc-annotation-final-resolution-v1",
      "reasoning_effort": "medium",
      "image_detail": "high",
      "images_per_case": 4,
      "max_output_tokens": 4000,
      "target_cases": 202,
      "pricing_mode": "batch",
      "endpoint": "/v1/responses",
      "completion_window": "24h",
      "max_jsonl_bytes": 190000000,
      "poll_interval_seconds": 30,
      "seed": 20261008,
      "minimum_visible_fraction": 0.7,
      "decisions": [
        "keep_with_justification",
        "regenerate"
      ],
      "uncertain_action": "regenerate",
      "apply_resoluti